# 06 - Evaluación y Explicabilidad del Modelo

**Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad**

- **Autor:** Jesús Goenaga Peña
- **Programa:** Doctorado en Ciencias Cognitivas - Universidad Autónoma de Manizales

---

Este notebook evalúa el modelo entrenado y aplica técnicas de explicabilidad:

| Análisis | Técnica | Propósito |
|----------|---------|----------|
| Métricas | Accuracy, F1, AUC, Precisión, Recall | Rendimiento cuantitativo |
| Curva ROC | Receiver Operating Characteristic | Sensibilidad vs especificidad |
| Matriz de confusión | Tabla de clasificación | Patrones de error |
| Grad-CAM | Mapas de saliencia | Regiones de interés por fase |
| Activaciones | Visualización de fases | Correspondencia neurofisiológica |

## 1. Configuración

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json, random
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
import glob
from torch.utils.data import DataLoader

PROJECT_ROOT = '/content/drive/MyDrive/cognitive-depth-model'
REPO_DIR = '/content/cognitive-depth-model'
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/yisusgoenaga/cognitive-depth-model.git {REPO_DIR}

result = subprocess.run(['find', REPO_DIR, '-name', 'cognitive_model.py'], capture_output=True, text=True)
model_path = result.stdout.strip()
src_dir = os.path.dirname(os.path.dirname(model_path))
sys.path.insert(0, src_dir)

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

## 2. Cargar Modelo Entrenado y Datos

In [ ]:
from model.cognitive_model import CognitiveDepthModel

# Recrear modelo con la misma configuración del entrenamiento
model = CognitiveDepthModel(
    in_channels=6,
    ngl_magno=8, ngl_parvo=8,
    v1_channels=16, v2_channels=32,
    v3_channels=64, v4_channels=64,
    v5_channels=128, integration_units=128,
    dropout=0.3,
).to(device)

# Cargar pesos del mejor modelo
best_path = os.path.join(CHECKPOINT_DIR, 'best_model_stage2.pth')
if not os.path.exists(best_path):
    best_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')

checkpoint = torch.load(best_path, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Modelo cargado desde: {os.path.basename(best_path)}')
print(f'Época: {checkpoint["epoch"]}, Test loss: {checkpoint["test_loss"]:.4f}')

In [ ]:
# Recrear el dataset de test
KITTI_RAW = os.path.join(PROJECT_ROOT, 'data/raw/kitti')
SPLITS_FILE = os.path.join(PROJECT_ROOT, 'data/splits/kitti_splits.json')

with open(SPLITS_FILE) as f:
    splits = json.load(f)

TRAIN_LEFT = os.path.join(KITTI_RAW, 'data_scene_flow/training/image_2')
TRAIN_RIGHT = os.path.join(KITTI_RAW, 'data_scene_flow/training/image_3')
TRAIN_DISP = os.path.join(KITTI_RAW, 'data_scene_flow/training/disp_occ_0')

class FixedKITTIDataset(torch.utils.data.Dataset):
    def __init__(self, scene_ids, left_dir, right_dir, disp_dir, target_size=(64, 128), augment=False):
        self.left_dir = left_dir
        self.right_dir = right_dir
        self.disp_dir = disp_dir
        self.target_size = target_size
        self.augment = augment
        self.scene_ids = [sid for sid in scene_ids
                          if os.path.exists(os.path.join(left_dir, f'{sid}.png'))]
        print(f'Dataset: {len(self.scene_ids)} escenas')

    def __len__(self):
        return len(self.scene_ids)

    def __getitem__(self, idx):
        import cv2, random, numpy as np
        sid = self.scene_ids[idx]
        h, w = self.target_size
        img_l = cv2.imread(os.path.join(self.left_dir, f'{sid}.png'))
        img_r = cv2.imread(os.path.join(self.right_dir, f'{sid}.png'))
        img_l = cv2.resize(img_l, (w, h)).astype(np.float32) / 255.0
        img_r = cv2.resize(img_r, (w, h)).astype(np.float32) / 255.0
        disp_path = os.path.join(self.disp_dir, f'{sid}.png')
        if os.path.exists(disp_path):
            disp = cv2.imread(disp_path, cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
            disp = cv2.resize(disp, (w, h))
            left_half = disp[:, :w//2]
            right_half = disp[:, w//2:]
            vl = left_half[left_half > 0]
            vr = right_half[right_half > 0]
            label = 1.0 if (len(vl) > 0 and len(vr) > 0 and vl.mean() > vr.mean()) else 0.0
        else:
            label = 0.5
        left_t = torch.from_numpy(img_l).permute(2, 0, 1)
        right_t = torch.from_numpy(img_r).permute(2, 0, 1)
        stereo = torch.cat([left_t, right_t], dim=0)
        return stereo, torch.tensor([label], dtype=torch.float32)

test_dataset = FixedKITTIDataset(splits['kitti_test'], TRAIN_LEFT, TRAIN_RIGHT, TRAIN_DISP)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

# También cargar train para comparación
train_dataset = FixedKITTIDataset(splits['kitti_train'], TRAIN_LEFT, TRAIN_RIGHT, TRAIN_DISP)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=False, num_workers=0)

print(f'Test: {len(test_loader)} batches | Train: {len(train_loader)} batches')

## 3. Evaluación Cuantitativa Completa

Métricas según la sección 8.4.1.4 de la tesis:
precisión, recall, F1-score, AUC-ROC.

In [ ]:
from evaluation.explainability import full_evaluation, plot_evaluation_summary

# Evaluar en test
print('Evaluando modelo en conjunto de PRUEBA...')
test_results = full_evaluation(model, test_loader, device)

print('\n' + '=' * 60)
print('MÉTRICAS DEL MODELO EN CONJUNTO DE PRUEBA')
print('=' * 60)
print(f'  Accuracy:  {test_results["accuracy"]:.4f}')
print(f'  Precision: {test_results["precision"]:.4f}')
print(f'  Recall:    {test_results["recall"]:.4f}')
print(f'  F1 Score:  {test_results["f1"]:.4f}')
print(f'  AUC-ROC:   {test_results["auc"]:.4f}')
print(f'\nReporte de clasificación:')
print(test_results['classification_report'])

In [ ]:
# Evaluar en train para comparar
print('Evaluando modelo en conjunto de ENTRENAMIENTO...')
train_results = full_evaluation(model, train_loader, device)

print('\n' + '=' * 60)
print('COMPARACIÓN TRAIN vs TEST')
print('=' * 60)
print(f'{"Métrica":<15} {"Train":>10} {"Test":>10} {"Diferencia":>12}')
print('-' * 50)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    tr = train_results[metric]
    te = test_results[metric]
    diff = tr - te
    print(f'{metric:<15} {tr:>10.4f} {te:>10.4f} {diff:>+12.4f}')

In [ ]:
# Visualización: ROC, Matriz de Confusión, Distribución de predicciones
plot_evaluation_summary(
    test_results,
    save_path=os.path.join(PROJECT_ROOT, 'results/visualizations/evaluation_summary.png')
)
print('Guardado en results/visualizations/evaluation_summary.png')

## 4. Grad-CAM: Regiones de Interés por Fase Cortical

Grad-CAM nos permite ver **qué regiones de la imagen** influyen en la decisión del modelo
en cada fase del procesamiento visual. Esto es crucial para:

- Establecer correspondencias funcionales con áreas del sistema visual (V1, V2, V3A, V5/MT)
- Verificar que las fases de disparidad (19, 26) atienden a diferencias entre ojos
- Confirmar que las fases de movimiento (25) detectan gradientes espacio-temporales
- Documentar la explicabilidad del modelo (sección 5.3.7 de la tesis)

In [ ]:
from evaluation.explainability import GradCAM, visualize_gradcam

# Tomar una imagen de ejemplo del test set
sample_input, sample_label = test_dataset[0]
sample_input = sample_input.unsqueeze(0).to(device)  # (1, 6, H, W)

# Mostrar la imagen de entrada
img_show = sample_input[0, :3].permute(1, 2, 0).cpu().numpy()
img_show = np.clip(img_show[:, :, ::-1], 0, 1)

with torch.no_grad():
    pred = model(sample_input)

print(f'Imagen de prueba - Label real: {sample_label.item():.0f}, '
      f'Predicción: {pred.item():.3f} ({"Más cercano" if pred.item() > 0.5 else "Más lejano"})')

fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.imshow(img_show)
ax.set_title(f'Imagen de entrada (ojo izquierdo)\n'
             f'Real: {"Más cercano" if sample_label.item() == 1 else "Más lejano"} | '
             f'Predicción: {"Más cercano" if pred.item() > 0.5 else "Más lejano"} ({pred.item():.3f})')
ax.axis('off')
plt.show()

In [ ]:
# Aplicar Grad-CAM en fases clave
# Necesitamos identificar las capas correctas del modelo
target_layers = {
    'Fase 12: V1 II/III': model.v1.layers_ii_iii[-1],
    'Fase 18: V2 Intercaladas': model.v2.interstripes[-1],
    'Fase 21: V3 Formas 3D': model.v3.v3_shapes[-1],
    'Fase 27: V5/MT Mapas': model.v5mt.dynamic_maps[-1],
}

visualize_gradcam(
    model, sample_input, target_layers,
    save_path=os.path.join(PROJECT_ROOT, 'results/grad_cam/gradcam_phases.png')
)
print('Grad-CAM guardado en results/grad_cam/gradcam_phases.png')

In [ ]:
# Grad-CAM en más ejemplos del test set
n_examples = 3
fig, axes = plt.subplots(n_examples, 5, figsize=(22, 4 * n_examples))
fig.suptitle('Grad-CAM en Fase V5/MT (Mapas 3D Dinámicos) - Múltiples Ejemplos',
             fontsize=14, fontweight='bold')

target_layer = model.v5mt.dynamic_maps[-1]

for i in range(n_examples):
    idx = i * (len(test_dataset) // n_examples)
    inp, lbl = test_dataset[idx]
    inp = inp.unsqueeze(0).to(device)

    # Imagen original
    img = inp[0, :3].permute(1, 2, 0).cpu().numpy()
    img = np.clip(img[:, :, ::-1], 0, 1)

    # Predicción
    with torch.no_grad():
        pred_val = model(inp).item()

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Ojo Izquierdo\nReal: {lbl.item():.0f}')
    axes[i, 0].axis('off')

    # Ojo derecho
    img_r = inp[0, 3:].permute(1, 2, 0).cpu().numpy()
    img_r = np.clip(img_r[:, :, ::-1], 0, 1)
    axes[i, 1].imshow(img_r)
    axes[i, 1].set_title(f'Ojo Derecho\nPred: {pred_val:.3f}')
    axes[i, 1].axis('off')

    # Grad-CAM
    try:
        grad_cam = GradCAM(model, target_layer)
        cam = grad_cam.generate(inp)
        cam_resized = cv2.resize(cam, (img.shape[1], img.shape[0]))

        axes[i, 2].imshow(cam_resized, cmap='jet')
        axes[i, 2].set_title('Grad-CAM (V5/MT)')
        axes[i, 2].axis('off')

        heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
        overlay = 0.6 * img + 0.4 * (heatmap[:, :, ::-1].astype(np.float32) / 255.0)
        axes[i, 3].imshow(np.clip(overlay, 0, 1))
        axes[i, 3].set_title('Superposición Izq')
        axes[i, 3].axis('off')

        overlay_r = 0.6 * img_r + 0.4 * (heatmap[:, :, ::-1].astype(np.float32) / 255.0)
        axes[i, 4].imshow(np.clip(overlay_r, 0, 1))
        axes[i, 4].set_title('Superposición Der')
        axes[i, 4].axis('off')
    except Exception as e:
        axes[i, 2].text(0.5, 0.5, str(e)[:60], ha='center', transform=axes[i, 2].transAxes)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results/grad_cam/gradcam_multiple_examples.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Guardado en results/grad_cam/gradcam_multiple_examples.png')

## 5. Activaciones con Datos Reales (Post-Entrenamiento)

Comparación clave: las activaciones **antes del entrenamiento** (Notebook 04, ruido aleatorio)
vs **después del entrenamiento** (datos reales, patrones aprendidos).

In [ ]:
from evaluation.explainability import visualize_phase_activations_real

# Tomar un ejemplo del test set
sample_input, _ = test_dataset[5]
sample_input = sample_input.unsqueeze(0).to(device)

visualize_phase_activations_real(
    model, sample_input,
    save_path=os.path.join(PROJECT_ROOT, 'results/visualizations/phase_activations_trained.png')
)
print('Guardado en results/visualizations/phase_activations_trained.png')

## 6. Guardar Resultados

In [ ]:
import pandas as pd

# Guardar métricas en CSV
metrics_df = pd.DataFrame([{
    'dataset': 'test',
    'accuracy': test_results['accuracy'],
    'precision': test_results['precision'],
    'recall': test_results['recall'],
    'f1': test_results['f1'],
    'auc': test_results['auc'],
}, {
    'dataset': 'train',
    'accuracy': train_results['accuracy'],
    'precision': train_results['precision'],
    'recall': train_results['recall'],
    'f1': train_results['f1'],
    'auc': train_results['auc'],
}])

metrics_csv = os.path.join(PROJECT_ROOT, 'results/metrics/evaluation_metrics.csv')
metrics_df.to_csv(metrics_csv, index=False)
print(f'Métricas guardadas en: {metrics_csv}')
print(metrics_df.to_string(index=False))

In [ ]:
# Guardar predicciones individuales para análisis posterior
predictions_df = pd.DataFrame({
    'true_label': test_results['labels'],
    'predicted_prob': test_results['probabilities'],
    'predicted_class': test_results['predictions'],
    'correct': (test_results['labels'] == test_results['predictions']).astype(int),
})

pred_csv = os.path.join(PROJECT_ROOT, 'results/metrics/test_predictions.csv')
predictions_df.to_csv(pred_csv, index=False)
print(f'Predicciones guardadas en: {pred_csv}')
print(f'\nDistribución de aciertos:')
print(predictions_df['correct'].value_counts())

In [ ]:
print('=' * 60)
print('NOTEBOOK 06 COMPLETADO')
print('=' * 60)
print()
print('Evaluación del modelo:')
print(f'  Accuracy (test):  {test_results["accuracy"]:.4f}')
print(f'  F1 Score (test):  {test_results["f1"]:.4f}')
print(f'  AUC-ROC (test):   {test_results["auc"]:.4f}')
print()
print('Archivos generados:')
print('  results/visualizations/evaluation_summary.png')
print('  results/grad_cam/gradcam_phases.png')
print('  results/grad_cam/gradcam_multiple_examples.png')
print('  results/visualizations/phase_activations_trained.png')
print('  results/metrics/evaluation_metrics.csv')
print('  results/metrics/test_predictions.csv')
print()
print('Siguiente: Notebook 07 - Validación con las 200 tareas')
print('  Generación de las tareas, ejecución del modelo, CSVs para comparación con humanos')